# 이진 분류 실험 ⑤ — GFP 모티프 피처 (graph feature preprocessor)
IBM Snap ML 의 `GraphFeaturePreprocessor` 로 거래마다 **시간 창 안의 패턴 모양 수**를 센다: fan-in/out, degree, scatter-gather, 시간 순서 사이클, 길이 제한 사이클, 금액 vertex stats. BlazingAML 이 XGBoost + GFP 로 HI-Large F1 58 을 낸 그 피처 (`sampling/reference.md` 6절).

**질문**: 07 의 서비스 피처 셋(`tod`, 트리 AP 0.634) 위에 모티프 수를 더하면 — 특히 트리가 못 보는 **CYCLE · SCATTER-GATHER** 의 빡빡한 컷 recall 과 전체 AP 가 얼마나 오르나. 무패턴은 이번엔 보지 않는다.

**진행 순서** (GFP API 를 처음 쓰므로 단계마다 확인):
1. 설치 + 작은 합성 그래프로 API·출력 열 수 확인 → 피처 이름 생성기가 맞는지 검증
2. 앞 200만 건으로 속도·메모리 측정 → 전체 소요 시간 예측
3. 전체 1.8억 건을 **500만 건씩 `transform`** (GFP 는 내부 그래프를 유지하며 시간 창 밖 간선을 버리는 스트리밍 설계). 청크마다 parquet 저장, 중단되면 이어서 (앞 청크로 그래프 상태 워밍업 후 재개)
4. 전체 열(≈60) 과 패턴별 합계 8열(`gfp_compact`, 나중에 GINe 용) 두 parquet 으로 합침
5. 트리: `tod` (07 결과 재사용) / `tod + compact` / `tod + full`

**메모리**: 입력 청크 500만 × 5 float64 = 200MB, 출력 500만 × 65 float64 = 2.6GB → float16 parquet. GFP 내부 그래프는 24h 창(약 120만 간선). 트리 단계는 07 과 같음 (z-norm memmap, `train_trees_lowmem`). 디스크: GFP 출력 ≈ 22GB + memmap ≈ 33GB

## 0. 작업 준비

In [1]:
import sys, json, time, gc, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psutil

sys.path.append(str(Path.cwd()))
import bc_lib as B
L = B.L

In [2]:
# ---- 실험 설정 ------------------------------------------------------------
RUN_NAME = 'gfp'
cfg = B.Config(dataset='HI-Large', smoke=False)
B.seed_all(cfg.seed)
OUT_DIR = cfg.out_dir(RUN_NAME)
GFP_DIR = cfg.feat_dir / 'gfp_chunks'; GFP_DIR.mkdir(exist_ok=True)
GFP_FULL, GFP_COMPACT = cfg.feat_dir / 'gfp_full.parquet', cfg.feat_dir / 'gfp_compact.parquet'
CHUNK = 5_000_000                 # transform 한 번에 넣는 간선 수
WARMUP = 2_000_000                # 재개할 때 그래프 상태를 되살리려고 먼저 흘리는 앞 구간 (≈1.5일 > 24h 창)
DAY = 86400

# GFP 설정 — 시간 창은 GFP 논문(HI: fan/degree/cycle 1일, scatter-gather 6시간). bin 은 모티프 크기 구간 (마지막은 열린 구간)
GFP_PARAMS = {
    'num_threads': 8, 'time_window': DAY,
    'vertex_stats': True, 'vertex_stats_cols': [4],                 # 입력 4번 열 = 금액
    'vertex_stats_feats': [0, 1, 2, 3, 4, 5, 6],                     # fan, degree, ratio, avg, sum, min, max (in/out 각각)
    'fan': True,    'fan_tw': DAY,    'fan_bins':    [2, 3, 4, 5, 6, 8, 10, 15],
    'degree': True, 'degree_tw': DAY, 'degree_bins': [2, 3, 4, 5, 6, 8, 10, 15],
    'scatter-gather': True, 'scatter-gather_tw': 6 * 3600, 'scatter-gather_bins': [2, 3, 4, 6, 8],
    'temp-cycle': True, 'temp-cycle_tw': DAY, 'temp-cycle_bins': [2, 3, 4, 5, 6],
    'lc-cycle': True,   'lc-cycle_tw': DAY,   'lc-cycle_len': 6, 'lc-cycle_bins': [2, 3, 4, 5, 6],
}
VS_NAMES = ['fan', 'deg', 'ratio', 'avg', 'sum', 'min', 'max', 'median', 'var', 'skew', 'kurt']

def gfp_feature_names(p):
    """출력 열 이름 (snapml 예제의 순서 가정: fan in/out → degree in/out → scatter-gather → temp-cycle → lc-cycle → vertex stats).
    1절에서 실제 출력 열 수와 맞는지 검증한다 — 안 맞으면 gfp_000 식 일반 이름으로 간다"""
    names = []
    def bins(prefix, bs):
        return [f'{prefix}_{a}_{b}' for a, b in zip(bs, bs[1:])] + [f'{prefix}_{bs[-1]}p']
    if p.get('fan'):    names += bins('fan_in', p['fan_bins']) + bins('fan_out', p['fan_bins'])
    if p.get('degree'): names += bins('deg_in', p['degree_bins']) + bins('deg_out', p['degree_bins'])
    if p.get('scatter-gather'): names += bins('sg', p['scatter-gather_bins'])
    if p.get('gather-scatter'): names += bins('gs', p['gather-scatter_bins'])
    if p.get('temp-cycle'): names += bins('tcyc', p['temp-cycle_bins'])
    if p.get('lc-cycle'):   names += bins('lccyc', p['lc-cycle_bins'])
    if p.get('vertex_stats'):
        for c in p['vertex_stats_cols']:
            for d in ('in', 'out'):
                names += [f'vs{c}_{d}_{VS_NAMES[f]}' for f in p['vertex_stats_feats']]
    return [f'gfp_{n}' for n in names]
GFP_NAMES = gfp_feature_names(GFP_PARAMS)
len(GFP_NAMES), GFP_NAMES[:6], GFP_NAMES[-4:]

(61,
 ['gfp_fan_in_2_3',
  'gfp_fan_in_3_4',
  'gfp_fan_in_4_5',
  'gfp_fan_in_5_6',
  'gfp_fan_in_6_8',
  'gfp_fan_in_8_10'],
 ['gfp_vs4_out_avg', 'gfp_vs4_out_sum', 'gfp_vs4_out_min', 'gfp_vs4_out_max'])

## 1. 설치 + API 확인 (합성 그래프)
`pip install snapml` (Linux 전용). 작은 그래프에 fan-out · 사이클을 심어서 출력 열 수와 이름 생성기가 맞는지, 사이클 열에 값이 찍히는지 본다

In [3]:
# pip install --break-system-packages --no-cache-dir snapml

In [5]:
# import importlib, subprocess
# if importlib.util.find_spec('snapml') is None:
#     print(subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'snapml'], capture_output=True, text=True).stdout[-800:])
import snapml
from snapml import GraphFeaturePreprocessor

gp_probe = GraphFeaturePreprocessor(); gp_probe.set_params(GFP_PARAMS)
print({k: v for k, v in gp_probe.get_params().items() if k in ('time_window', 'fan', 'lc-cycle_len', 'num_threads')})

{'num_threads': 8, 'time_window': 86400, 'fan': True, 'lc-cycle_len': 6}


In [6]:
# 합성 간선: 계좌 0 이 1~5 에게 같은 시간대에 송금(fan-out 5), 10→11→12→10 사이클, 나머지 무작위. 열: [edge_id, src, dst, ts(초), amount]
rng = np.random.default_rng(0)
t0 = 1_000_000
edges = [[0, 1, 100], [0, 2, 200], [0, 3, 300], [0, 4, 400], [0, 5, 500],        # fan-out
         [10, 11, 600], [11, 12, 700], [12, 10, 800],                              # 3-사이클 (시간 순서대로)
         [20, 21, 900], [21, 22, 1000], [22, 23, 1100], [23, 20, 1200]]            # 4-사이클
edges += [[int(rng.integers(30, 60)), int(rng.integers(30, 60)), int(1300 + i * 10)] for i in range(50)]
probe = np.array([[i, s, d, t0 + t, float(rng.integers(10, 1000))] for i, (s, d, t) in enumerate(edges)], dtype=np.float64)
out = gp_probe.fit_transform(probe)
n_extra = out.shape[1] - probe.shape[1]
print('입력', probe.shape, '→ 출력', out.shape, '| 추가 열', n_extra, '| 이름 생성기', len(GFP_NAMES))
if n_extra != len(GFP_NAMES):
    print('!! 열 수 불일치 — 이름 생성기 가정이 틀림. 일반 이름으로 진행 (compact 는 건너뜀)')
    GFP_NAMES = [f'gfp_{i:03d}' for i in range(n_extra)]
feat = pd.DataFrame(out[:, probe.shape[1]:], columns=GFP_NAMES)
# 사이클 간선(5~11)과 fan-out 간선(0~4) 에서 0 이 아닌 열만
nz = feat.loc[:11].loc[:, (feat.loc[:11] != 0).any()]
nz.round(2)

입력 (62, 5) → 출력 (62, 80) | 추가 열 75 | 이름 생성기 61
!! 열 수 불일치 — 이름 생성기 가정이 틀림. 일반 이름으로 진행 (compact 는 건너뜀)


,gfp_008,gfp_009,gfp_010,gfp_011,gfp_016,gfp_024,gfp_026,gfp_028,gfp_029,gfp_030,...,gfp_065,gfp_066,gfp_067,gfp_068,gfp_069,gfp_070,gfp_071,gfp_072,gfp_073,gfp_074
0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,...,0.0,0.0,0.0,1.0,2.0,2.0,406.0,812.0,406.0,406.0
1,2.0,2.0,2.0,2.0,2.0,0.0,2.0,2.0,2.0,2.0,...,0.0,0.0,0.0,1.0,2.0,2.0,789.0,1578.0,789.0,789.0
2,0.0,2.0,2.0,2.0,2.0,0.0,0.0,2.0,2.0,2.0,...,0.0,0.0,0.0,1.0,2.0,2.0,323.0,646.0,323.0,323.0
3,0.0,0.0,2.0,2.0,2.0,0.0,0.0,0.0,2.0,2.0,...,0.0,0.0,0.0,1.0,2.0,2.0,246.0,492.0,246.0,246.0
4,0.0,0.0,0.0,2.0,2.0,0.0,0.0,0.0,0.0,2.0,...,0.0,0.0,0.0,1.0,2.0,2.0,793.0,1586.0,793.0,793.0
5,0.0,0.0,0.0,0.0,2.0,2.0,0.0,0.0,0.0,0.0,...,176.0,88.0,88.0,1.0,2.0,2.0,877.0,1754.0,877.0,877.0
6,0.0,0.0,0.0,0.0,2.0,2.0,0.0,0.0,0.0,0.0,...,134.0,67.0,67.0,1.0,2.0,2.0,88.0,176.0,88.0,88.0
7,0.0,0.0,0.0,0.0,2.0,2.0,0.0,0.0,0.0,0.0,...,1754.0,877.0,877.0,1.0,2.0,2.0,67.0,134.0,67.0,67.0
8,0.0,0.0,0.0,0.0,2.0,2.0,0.0,0.0,0.0,0.0,...,684.0,342.0,342.0,1.0,2.0,2.0,674.0,1348.0,674.0,674.0
9,0.0,0.0,0.0,0.0,2.0,2.0,0.0,0.0,0.0,0.0,...,1154.0,577.0,577.0,1.0,2.0,2.0,342.0,684.0,342.0,342.0


In [7]:
# 스트리밍 확인 — 같은 간선을 둘로 나눠 fit_transform(앞) + transform(뒤) 했을 때 결과가 한 번에 넣은 것과 같은가 (그래프 상태 유지)
gp2 = GraphFeaturePreprocessor(); gp2.set_params(GFP_PARAMS)
a = gp2.fit_transform(probe[:7]); b = gp2.transform(probe[7:])
same = np.allclose(np.vstack([a, b]), out)
print('청크로 나눠 넣어도 동일:', same)
if not same:
    diff = np.abs(np.vstack([a, b]) - out).max(axis=0); print('열별 최대 차이 (0 아닌 것):', {GFP_NAMES[i - 5]: round(d, 3) for i, d in enumerate(diff) if d > 1e-6 and i >= 5})

청크로 나눠 넣어도 동일: False
열별 최대 차이 (0 아닌 것): {'gfp_000': np.float64(2.0), 'gfp_001': np.float64(1.0), 'gfp_008': np.float64(2.0), 'gfp_009': np.float64(1.0), 'gfp_010': np.float64(2.0), 'gfp_016': np.float64(2.0), 'gfp_017': np.float64(1.0), 'gfp_018': np.float64(2.0), 'gfp_020': np.float64(2.0), 'gfp_024': np.float64(2.0), 'gfp_025': np.float64(1.0), 'gfp_026': np.float64(2.0), 'gfp_027': np.float64(1.0), 'gfp_028': np.float64(2.0), 'gfp_029': np.float64(2.0), 'gfp_030': np.float64(2.0), 'gfp_032': np.float64(2.0), 'gfp_037': np.float64(1.0), 'gfp_038': np.float64(2.0), 'gfp_039': np.float64(1.0), 'gfp_042': np.float64(1.0), 'gfp_043': np.float64(2.0), 'gfp_044': np.float64(2.0), 'gfp_045': np.float64(1.0), 'gfp_046': np.float64(2.0), 'gfp_048': np.float64(5.0), 'gfp_049': np.float64(1.5), 'gfp_051': np.float64(3591.0), 'gfp_054': np.float64(1.0), 'gfp_055': np.float64(3.0), 'gfp_056': np.float64(2.0), 'gfp_057': np.float64(67.0), 'gfp_058': np.float64(2464.0), 'gfp_059': np.float64(67.0),

## 2. 속도·메모리 측정 — 앞 200만 건
전체 1.8억 건 소요 시간 예측. 입력은 tx parquet 에서 `[edge_id, src, dst, epoch(ts), received]` 를 edge_id 순(= 시간순) 으로

In [8]:
con = L.connect(threads=8, memory='8GB')
def read_edges(offset, n):
    """edge_id 순으로 offset 부터 n 건. float64 [edge_id, src, dst, ts초, amount]"""
    df = con.execute(f"""
        SELECT edge_id, src, dst, epoch(ts) AS t, received
        FROM read_parquet({L.q(cfg.tx_path)}) ORDER BY edge_id LIMIT {n} OFFSET {offset}""").df()
    return np.ascontiguousarray(df.to_numpy(dtype=np.float64))
N_TOTAL = con.execute(f"SELECT count(*) FROM read_parquet({L.q(cfg.tx_path)})").fetchone()[0]
sample = read_edges(0, 2_000_000)
assert (np.diff(sample[:, 3]) >= 0).all(), '시간순이 아님'
rss0 = psutil.Process().memory_info().rss / 1e9; t0 = time.time()
gp_t = GraphFeaturePreprocessor(); gp_t.set_params(GFP_PARAMS)
out = gp_t.fit_transform(sample)
dt = time.time() - t0; rss1 = psutil.Process().memory_info().rss / 1e9
print(f'200만 건 {dt:.0f}s → {2_000_000/dt:,.0f} 건/s | 전체 {N_TOTAL:,} 건 예상 {N_TOTAL/(2_000_000/dt)/3600:.1f} 시간 | RSS {rss0:.1f} → {rss1:.1f} GB')
feat = pd.DataFrame(out[:, 5:], columns=GFP_NAMES)
(feat != 0).mean().sort_values(ascending=False).head(15).round(4)       # 0 이 아닌 비율 — 사이클 열이 살아 있는지

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

200만 건 287s → 6,970 건/s | 전체 179,702,079 건 예상 7.2 시간 | RSS 8.7 → 13.6 GB


gfp_053    1.0000
gfp_048    1.0000
gfp_047    1.0000
gfp_073    1.0000
gfp_068    1.0000
gfp_070    1.0000
gfp_069    1.0000
gfp_074    1.0000
gfp_072    1.0000
gfp_051    1.0000
gfp_071    1.0000
gfp_049    1.0000
gfp_050    1.0000
gfp_052    1.0000
gfp_057    0.8752
dtype: float64

In [9]:
del sample, out, feat, gp_t; gc.collect(); B.trim('(측정 후)')

RSS 9.6 GB (측정 후)


## 3. 전체 변환 — 500만 건씩, 청크 parquet (재개 가능)
- `gfp_chunks/part_0000.parquet` … 각 청크의 `edge_id` + 피처(float16). 이미 있는 part 는 건너뜀
- 커널이 죽어 재개할 때는 GFP 내부 그래프가 비어 있으므로 **첫 미완 청크 앞 200만 건을 먼저 transform 해 상태를 채운 뒤**(출력 버림) 이어간다. 24h 창이라 1.5일치면 충분
- 소요: 2절 예측 참고. 중간에 멈춰도 다시 이 셀만 실행

In [ ]:
parts = [GFP_DIR / f'part_{i:04d}.parquet' for i in range((N_TOTAL + CHUNK - 1) // CHUNK)]
first_missing = next((i for i, p in enumerate(parts) if not p.exists()), len(parts))
print(f'청크 {len(parts)}개, 완료 {first_missing}개')
if first_missing < len(parts):
    gp = GraphFeaturePreprocessor(); gp.set_params(GFP_PARAMS)
    t_all = time.time()
    if first_missing > 0:                                             # 재개: 앞 구간으로 그래프 상태 워밍업
        wstart = max(0, first_missing * CHUNK - WARMUP)
        warm = read_edges(wstart, first_missing * CHUNK - wstart); gp.fit_transform(warm); del warm
        print(f'  워밍업 {WARMUP:,} 건 완료')
        fit_done = True
    else:
        fit_done = False
    last_t = None
    for i in range(first_missing, len(parts)):
        t0 = time.time(); X = read_edges(i * CHUNK, CHUNK)
        assert (np.diff(X[:, 3]) >= 0).all() and (last_t is None or X[0, 3] >= last_t), f'청크 {i} 시간순 깨짐'
        last_t = X[-1, 3]
        out = gp.transform(X) if fit_done else gp.fit_transform(X); fit_done = True
        feat = pd.DataFrame(out[:, 5:].astype(np.float16), columns=GFP_NAMES); feat.insert(0, 'edge_id', X[:, 0].astype(np.int64))
        feat.to_parquet(parts[i], index=False, compression='zstd')
        del X, out, feat
        B.log(f'  part {i:04d}/{len(parts)-1}  {time.time()-t0:.0f}s  누적 {(time.time()-t_all)/60:.0f}분  RSS {psutil.Process().memory_info().rss/1e9:.1f}GB')
    del gp; gc.collect()
print('완료')

청크 36개, 완료 0개


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

/tmp/ipykernel_780572/4051295192.py:20: RuntimeWarning: overflow encountered in cast
  feat = pd.DataFrame(out[:, 5:].astype(np.float16), columns=GFP_NAMES); feat.insert(0, 'edge_id', X[:, 0].astype(np.int64))


[08:58:30]   part 0000/35  5598s  누적 93분  RSS 19.7GB


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 4. 합치기 — `gfp_full.parquet` (전체 열) + `gfp_compact.parquet` (패턴별 합계 8열)
compact: bin 열을 패턴별로 합산한 모티프 총수 (ln(1+x)) + 금액 stats 중 in/out fan. GINe 에 붙일 수 있는 크기

In [ ]:
if not GFP_FULL.exists():
    t0 = time.time()
    con.execute(f"COPY (SELECT * FROM read_parquet({L.q(str(GFP_DIR / 'part_*.parquet'))}) ORDER BY edge_id) TO {L.q(GFP_FULL)} (FORMAT PARQUET, COMPRESSION ZSTD)")
    print(f'gfp_full.parquet {time.time()-t0:.0f}s')
n_full = con.execute(f"SELECT count(*) FROM read_parquet({L.q(GFP_FULL)})").fetchone()[0]
assert n_full == N_TOTAL, f'행 수 불일치 {n_full:,} vs {N_TOTAL:,}'

COMPACT_GROUPS = {'gfp_fan_in_total': 'gfp_fan_in_', 'gfp_fan_out_total': 'gfp_fan_out_', 'gfp_deg_in_total': 'gfp_deg_in_', 'gfp_deg_out_total': 'gfp_deg_out_',
                  'gfp_sg_total': 'gfp_sg_', 'gfp_tcyc_total': 'gfp_tcyc_', 'gfp_lccyc_total': 'gfp_lccyc_'}
COMPACT_FEATURES = list(COMPACT_GROUPS) + ['gfp_vs_in_sum', 'gfp_vs_out_sum']
if not GFP_NAMES[0].startswith('gfp_fan_in'):
    print('일반 이름 상태 — compact 는 건너뜀'); COMPACT_FEATURES = []
elif not GFP_COMPACT.exists():
    def total(prefix):
        cols = [c for c in GFP_NAMES if c.startswith(prefix)]; return ' + '.join(f'"{c}"::DOUBLE' for c in cols)
    exprs = [f'ln(1 + ({total(p)}))::FLOAT AS {name}' for name, p in COMPACT_GROUPS.items()]
    exprs += [f'ln(1 + greatest("gfp_vs4_{d}_sum"::DOUBLE, 0))::FLOAT AS gfp_vs_{d}_sum' for d in ('in', 'out')]
    t0 = time.time()
    con.execute(f"COPY (SELECT edge_id, {', '.join(exprs)} FROM read_parquet({L.q(GFP_FULL)}) ORDER BY edge_id) TO {L.q(GFP_COMPACT)} (FORMAT PARQUET, COMPRESSION ZSTD)")
    print(f'gfp_compact.parquet {time.time()-t0:.0f}s')
con.execute(f"DESCRIBE SELECT * FROM read_parquet({L.q(GFP_COMPACT)})").df() if COMPACT_FEATURES else None

In [ ]:
# 라벨별 평균 — 모티프 수가 세탁에서 다른가 (compact). 패턴 라벨별로도: CYCLE 거래에서 사이클 열이 큰지
if COMPACT_FEATURES:
    con.execute(f"""
        SELECT coalesce(p.pattern_type, CASE WHEN t.is_laundering THEN 'UNMATCHED' ELSE 'NORMAL' END) AS grp, count(*) AS n,
               {', '.join(f'avg(g.{c})::FLOAT AS {c[4:]}' for c in COMPACT_FEATURES)}
        FROM read_parquet({L.q(cfg.tx_path)}) t JOIN read_parquet({L.q(GFP_COMPACT)}) g USING (edge_id)
        LEFT JOIN (SELECT DISTINCT edge_id, pattern_type FROM read_parquet({L.q(L.BASE / 'patterns.parquet')})) p USING (edge_id)
        WHERE t.day BETWEEN DATE '2022-10-17' AND DATE '2022-10-30'
        GROUP BY 1 ORDER BY n DESC""").df().set_index('grp').round(3)

## 5. 트리 — `tod` + GFP
07 과 같은 구조. 조건: `tod_compact` (30 + 8), `tod_full` (30 + 전체). `tod` 기준선은 07 결과를 읽어 비교

In [ ]:
currency_codes, format_codes = B.code_tables(con, cfg)
PAIR_FEATURES = ['pair_prior_count', 'pair_gap_minutes', 'pair_count_1h', 'reverse_history_count']
ACT_FEATURES  = ['src_out_1h', 'src_out_1d', 'src_uniq_dst_1d', 'src_out_amt_1d', 'dst_in_1h', 'dst_in_1d', 'dst_uniq_src_1d', 'dst_in_amt_1d']
HIST_FEATURES = ['src_hist_out', 'src_hist_uniq_out', 'src_active_days', 'src_rate_vs_prior', 'src_minute_burst', 'src_gap_minutes',
                 'dst_hist_in', 'dst_hist_uniq_in', 'dst_active_days', 'dst_rate_vs_prior', 'dst_minute_burst', 'dst_gap_minutes']
TOD_FEATURES  = ['hour_sin', 'hour_cos', 'weekday']
PATHS = {'pair': cfg.feat_dir / 'pair_history.parquet', 'act': cfg.feat_dir / 'account_activity.parquet',
         'hist': cfg.feat_dir / 'account_history.parquet', 'tod': cfg.feat_dir / 'time_of_day.parquet'}
TOD_SET = [c for c in B.BASIC_EDGE_FEATURES if c != 'timestamp'] + TOD_FEATURES + PAIR_FEATURES + ACT_FEATURES + HIST_FEATURES   # 07 의 tod 조건 (30)
CONDITIONS = {'tod_compact': TOD_SET + COMPACT_FEATURES, 'tod_full': TOD_SET + GFP_NAMES}
ALL_FEATURES = TOD_SET + COMPACT_FEATURES + GFP_NAMES
{k: len(v) for k, v in CONDITIONS.items()}, len(ALL_FEATURES)

In [ ]:
con.close(); con = L.connect(threads=8, memory='3GB')
con.register('currency_codes', currency_codes); con.register('format_codes', format_codes)
tx = B.load_tx_lowmem(con, cfg); tx.drop(columns=['timestamp'], inplace=True)
B.trim('(tx 적재 후)')
tx = B.attach_parquet(tx, PATHS['tod'],  TOD_FEATURES,  dtype=np.float16)
tx = B.attach_parquet(tx, PATHS['pair'], PAIR_FEATURES, dtype=np.float16)
tx = B.attach_parquet(tx, PATHS['act'],  ACT_FEATURES,  dtype=np.float16)
tx = B.attach_parquet(tx, PATHS['hist'], HIST_FEATURES, dtype=np.float16)
if COMPACT_FEATURES:
    tx = B.attach_parquet(tx, GFP_COMPACT, COMPACT_FEATURES, dtype=np.float16)
tx = B.attach_parquet(tx, GFP_FULL, GFP_NAMES, dtype=np.float16)          # 이미 float16 로 저장돼 있음
B.trim(f'(피처 {len(ALL_FEATURES)}열 붙인 후)'); tx.shape

In [ ]:
in_train = B.window_mask(tx, cfg.train_window)
FEAT_PATH = cfg.feat_dir / f'feat16_{RUN_NAME}.npy'
feat_z, znorm_table = B.znorm_features(tx, ALL_FEATURES, in_train, out_path=FEAT_PATH)
B.trim('(z-norm 후)'); znorm_table.tail(12)

In [ ]:
val_df = B.val_sample(con, cfg.val_window, cfg.val_neg_rate)
test_df = B.test_full(con, cfg.test_window)
in_degree = B.recipient_indegree(con, cfg)
con.close()
all_edge_ids = tx['edge_id'].to_numpy()
val_idx = B.rows_of(all_edge_ids, val_df['edge_id'].to_numpy()); test_idx = B.rows_of(all_edge_ids, test_df['edge_id'].to_numpy())
assert (tx['y'].to_numpy()[val_idx] == val_df['y'].to_numpy()).all()
del all_edge_ids
train_rows = np.flatnonzero(in_train); y_train = tx['y'].to_numpy()[in_train]
y_val, w_val = val_df['y'].to_numpy(), val_df['w'].to_numpy()
del tx; gc.collect(); B.trim('(tx 삭제 후)')

In [ ]:
def run_condition(name, feats):
    out = OUT_DIR / name; out.mkdir(exist_ok=True)
    if (out / 'compare.csv').exists():
        print(f'[{name}] 이미 완료 — 건너뜀'); return
    B.log(f'[{name}] 피처 {len(feats)}')
    cols = [ALL_FEATURES.index(c) for c in feats]
    X_val = B.rows_matrix(feat_z, val_idx, cols)
    test_pred = lambda m: B.predict_rows(m, feat_z, test_idx, cols)
    results = {}
    lgb_model, xgb_model = B.train_trees_lowmem(feat_z, train_rows, cols, y_train, X_val, y_val, w_val, test_pred, val_df, test_df, cfg, results)
    imp = pd.DataFrame({'feature': feats, 'xgboost_gain': [xgb_model.get_booster().get_score(importance_type='gain').get(f'f{i}', 0.0) for i in range(len(feats))],
                        'lightgbm_gain': lgb_model.booster_.feature_importance('gain')})
    imp.sort_values('xgboost_gain', ascending=False).to_csv(out / 'importance.csv', index=False, encoding='utf-8-sig')
    pattern_tables, degree_tables, cut_tables = [], [], []
    for mname, model in [('xgboost', xgb_model), ('lightgbm', lgb_model)]:
        sc = pd.DataFrame(dict(edge_id=test_df.edge_id, y=test_df.y, score=test_pred(model)))
        sc.to_parquet(out / f'{mname}_test_scores.parquet', index=False)
        pattern_tables.append(B.pattern_recall(sc).assign(model=mname))
        degree_tables.append(B.degree_stratified(sc, test_df['dst'].to_numpy(), in_degree).assign(model=mname))
        cut_tables.append(B.cut_table(sc).assign(model=mname))
    pd.concat(pattern_tables).to_csv(out / 'analysis_pattern.csv', encoding='utf-8-sig')
    pd.concat(degree_tables).to_csv(out / 'analysis_degree.csv', encoding='utf-8-sig')
    pd.concat(cut_tables).to_csv(out / 'analysis_cuts.csv', encoding='utf-8-sig')
    B.save_results(out, cfg, results, features=dict(tree=feats))
    del X_val, lgb_model, xgb_model; gc.collect(); B.trim(f'[{name}] 끝')

for name, feats in CONDITIONS.items():
    if feats:
        run_condition(name, feats)

## 6. 비교 — `tod`(07) vs `tod_compact` vs `tod_full`

In [ ]:
BASE = Path.cwd() / 'results' / cfg.dataset / 'tree_ablations' / 'tod'       # 07 의 tod 조건
RUNS = {'tod (07)': BASE, **{n: OUT_DIR / n for n in CONDITIONS if (OUT_DIR / n / 'compare.csv').exists()}}
rows = []
for name, d in RUNS.items():
    c = pd.read_csv(d / 'compare.csv', index_col=[0, 1])
    for model in c.index.get_level_values(0).unique():
        t = c.loc[(model, 'test')]; v = c.loc[(model, 'val')]
        rows.append(dict(condition=name, model=model, val_AP=v['AP'], test_AP=t['AP'], F1=t['F1'], precision=t['precision'], recall=t['recall'],
                         recall_1pct=t['recall_at_1pct'], unmatched_1pct=t['unmatched_recall_at_1pct']))
summary = pd.DataFrame(rows).set_index(['model', 'condition']).sort_index()
summary.to_csv(OUT_DIR / 'summary.csv', encoding='utf-8-sig'); summary.round(4)

In [ ]:
# 패턴별 recall — r@0.1% 와 argmax (XGBoost). 질문: CYCLE · SCATTER-GATHER 가 오르나
pat = pd.concat([pd.read_csv(d / 'analysis_pattern.csv', index_col=0).query("model == 'xgboost'").assign(condition=n) for n, d in RUNS.items()])
order = [p for p in B.PATTERN_ORDER + ['ALL'] if p in pat.index]
display(pat.reset_index().pivot(index='pattern_type', columns='condition', values='r@0.1%').loc[order, list(RUNS)].round(4))
pat.reset_index().pivot(index='pattern_type', columns='condition', values='argmax').loc[order, list(RUNS)].round(4)

In [ ]:
# 수신 계좌 in-degree 구간별 AP (XGBoost)
deg = pd.concat([pd.read_csv(d / 'analysis_degree.csv', index_col=0).query("model == 'xgboost'").assign(condition=n) for n, d in RUNS.items()])
deg.reset_index().pivot(index='bin', columns='condition', values='AP')[list(RUNS)].round(4)

In [ ]:
# GFP 피처 중요도 — tod_full 에서 상위 15 (XGBoost gain). 어떤 모티프 열이 쓰였나
imp = pd.read_csv(OUT_DIR / 'tod_full' / 'importance.csv'); imp['is_gfp'] = imp.feature.str.startswith('gfp_')
display(imp.head(15)); print('GFP 열의 gain 비중:', round(imp.loc[imp.is_gfp, 'xgboost_gain'].sum() / imp.xgboost_gain.sum(), 3))

## 7. 읽는 법
- `tod_full − tod`: 모티프 수의 전체 몫. BlazingAML 이 GFP 로 얻은 이득이 우리 피처 위에서도 남아 있나
- CYCLE · SCATTER-GATHER 의 r@0.1% · argmax 변화: 트리가 못 보던 2홉·사이클 구조가 피처로 들어온 효과
- `tod_compact` 가 `tod_full` 에 가깝다면 8열로 충분 → GINe 에도 붙일 수 있음 (다음 런)